In [ ]:
import numpy as np
import warp as wp
import pyglet
from utils.DiffXPBD import DiffXPBDTapeFramework3D_Warp
from utils.AdamOptimizer import AdamOptimizer
%load_ext autoreload
%autoreload 2

In [ ]:
wp.init()
print("CUDA available:", wp.is_cuda_available())
print("Preferred device:", wp.get_preferred_device())
DEVICE = "cuda:0" if wp.is_cuda_available() else "cpu"

In [ ]:
dt = 1.0e-2
total_time = 100
total_steps = int(total_time / dt)
MAX_EPOCHS = 1000
itr = 0
gripper_side = 'left' # 'right' or 'left'

mesh_file_path = f"mesh/ad_gripper_less_stiff.msh"

sim_target_path = "data/Youngs_9500000_time_200_dt_0.01_sweep_20_damping_amp_1.00e+00.npz"

contact_loc = "mid"
real_target_kp_path = f"data/kps_trajectories/{contact_loc}_contact/synced_kps.csv"
real_applied_force_path = f"data/kps_trajectories/{contact_loc}_contact/synced_forces.csv"
kp_init_path = "data/PosEffs_all_pts_left.json"
trajectory_type = "kps" # "kps" or "mesh_pts"
optimize_type = "sim2real" # "sim2sim" or "sim2real"

In [ ]:
if contact_loc == "up":
    contact_pos = (18.85565662, 0, 65.47258853)
elif contact_loc == "mid":
    contact_pos = (18.85565662, 0, 45.20056126)
elif contact_loc == "down":
    contact_pos = (18.85565662, 0, 24.92853398)

In [ ]:
solver = DiffXPBDTapeFramework3D_Warp(
    trajectory_type=trajectory_type,
    optimize_type=optimize_type,
    mesh_path=mesh_file_path,
    real_target_kps_path=real_target_kp_path,
    sim_target_npz=sim_target_path,
    target_unit="mm",
    dt=dt,
    gravity_vec=(0.,  -0., 0.),
    mass_total= 0.00729,
    # 0.00729,
    poisson_ratio=0.48,
    compliance_modulation=1.e-2,
    
    fixed_center=[(0.71246, 0, 4.03762)],
    fixed_rotation=[(0., 10.007068, 0.)],
    fixed_extents=[(20, 10, 5.5)],
    
    applied_center=[contact_pos],
    applied_extents=[(2.5, 10, 2)],
    constant_applied_force=(-5.0, 0.0, 0.0),
    show_force_arrow=True,
    series_force_path=real_applied_force_path,
    series_force_mode=True,
    
    camera_pos=(0., 180., 45.2006) if gripper_side == 'right' else (0., -180., 45.2006),
    camera_front=(0.0, -1.0, 0.0) if gripper_side == 'right' else (0.0, 1.0, 0.0),
    background_color=(0., 0., 0.),
    text_color = (0, 0, 0, 255) if gripper_side == 'right' else (255, 255, 255, 255),

    youngs_init = 95 * 1e6,
    force_amplification = 1.e0,
    damping_amplification=1/51625.07614318651,
    mass_amplification=51625.07614318651,
    #0.8838519448503126,
    sweep_count=20,

    position_effector_path = kp_init_path,

    device=DEVICE,)

In [ ]:
damping = (4.0 * dt * solver.youngs_np * solver.damping_amp_np) / ((1.0 + solver.poisson_ratio) * solver.avg_edge * solver.avg_edge)
damp_term = dt * damping
mass_term = solver.mass_per_vertex
print('Damping:', damping)
print('Total mass:', solver.mass_total)
print('Damping term:', damp_term)
print('Mass term:', mass_term)
print('Damping to mass ratio:', damp_term / mass_term)

In [ ]:
STOP_CONDITION = ["convergence", "thresholding", "both"]
OPTIMIZED_METHOD = [None, "simultaneous", "alternating"]
OPTIMIZE_SUBJECT = ["Youngs_Modulus", "Applied_Force", "Force_Amplification", "Damping_Amplification"]

optimized_subjects = [OPTIMIZE_SUBJECT[3], OPTIMIZE_SUBJECT[0]]
optimized_lr = [2.e-1, 1.e-2]

optimized_str = ""
for i in range(len(optimized_subjects)):
    optimized_str += f"{optimized_subjects[i]}"
    if i != len(optimized_subjects) - 1:
        optimized_str += "_"

gui_mode = True
if gui_mode:
    solver.run_opengl_viewer(total_steps=total_steps, 
                             total_time=int(total_time), 
                             save_frames=False, 
                             gradient_mode=False)
else:
    solver.train(project_name = f"{optimize_type}_{trajectory_type}_{contact_loc.capitalize()}_contact_Time_{int(total_time)}s_Max_Epochs_{int(MAX_EPOCHS)}_Optimized_{optimized_str}",
                contact_pos = contact_loc,
                stop_condition = STOP_CONDITION[2],
                convergence_patience = 10,
                relative_change_threshold = 1.0e-3,
                optimized_method = OPTIMIZED_METHOD[2],
                alternating_epochs = 100,
                max_epochs = MAX_EPOCHS, 
                lr = optimized_lr, 
                eps = 1.0e0, 
                optimize_subject = optimized_subjects,
                total_steps = total_steps,
                optimizer = "Adam")